# Program-1: Text Preprocessing using NLTK and spaCy

## Aim

To build and evaluate a text preprocessing pipeline that performs tokenization, normalization, stop-word removal, stemming, lemmatization, part-of-speech tagging, and named entity recognition.

## Problem Statement

**SMS Message Cleaning and Entity Audit:** A mobile-service quality team must standardize real SMS messages before spam analysis. Students clean the UCI SMS Spam Collection, compare NLTK and spaCy tokenization and lemmatization, create POS summaries, and inspect named entities and preprocessing errors without exposing personal messages.

## Software Requirements

- Python 3.x
- Jupyter Notebook or Google Colab
- NLTK
- spaCy and an English language model
- Pandas and regular expressions

## Suggested Dataset

UCI Machine Learning Repository — SMS Spam Collection (5,574 labelled SMS messages).

Official page: https://archive.ics.uci.edu/dataset/228/sms+spam+collection

## Dataset Access

Recommended:

```python
from ucimlrepo import fetch_ucirepo
sms = fetch_ucirepo(id=228)
```

Alternatively, download the ZIP from the official UCI page. No account is required.

## Recommended Lab Subset

Use the complete dataset for cleaning statistics; use a stratified sample of 500 messages for detailed NLTK–spaCy comparison.

## Procedure

### Step 1: Import and Setup

- Install or import NLTK, spaCy, Pandas, and regular-expression utilities.
- Download the required NLTK resources and load a spaCy language model.

### Step 2: Load and Inspect Text

- Load the selected corpus and report document count, average length, and sample records.
- Identify HTML, URLs, emojis, punctuation, case variation, and missing values.

### Step 3: Normalize Text

- Convert case consistently where appropriate.
- Remove or replace URLs, HTML tags, repeated whitespace, and unwanted symbols while retaining meaningful punctuation when needed.

### Step 4: Tokenize and Remove Stop Words

- Apply sentence and word tokenization using NLTK or spaCy.
- Remove stop words and compare token counts before and after filtering.

### Step 5: Apply Stemming and Lemmatization

- Generate Porter or Snowball stems.
- Generate context-aware lemmas and compare representative outputs.

### Step 6: Perform POS Tagging and NER

- Assign part-of-speech tags to each token.
- Extract named entities and group them by entity label.

### Step 7: Evaluate and Document

- Compare NLTK and spaCy outputs for at least five sentences.
- Discuss errors caused by ambiguity, abbreviations, domain terms, and code-switching.

## Expected Result

A reusable preprocessing pipeline with comparative observations for tokenization, stemming, lemmatization, POS tagging, and NER.

## Observations to Record

- Dataset size and split details.
- Preprocessing and model configuration.
- Evaluation metrics and sample outputs.
- Errors, limitations, ethical considerations, and proposed improvements.

## Viva Voice Questions

1. Why is lemmatization usually more linguistically meaningful than stemming?
2. When should stop words be retained?
3. How do POS tags support lemmatization?
4. What causes NER errors in domain-specific text?

In [ ]:
%pip -q install ucimlrepo nltk spacy pandas scikit-learn

In [ ]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 92.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# **Import the Required Libraries**

In [ ]:


import re
import html
import string
import pandas as pd
import nltk
import spacy

from collections import Counter
from ucimlrepo import fetch_ucirepo
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split

print("All libraries imported successfully!")

All libraries imported successfully!


# **Download NLTK Resources and Load the spaCy Model**

In [ ]:


nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')


nlp = spacy.load("en_core_web_sm")

print("NLTK resources downloaded successfully!")
print("spaCy English model loaded successfully!")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


NLTK resources downloaded successfully!
spaCy English model loaded successfully!


# **Load the SMS Spam Collection Dataset**

In [ ]:
response = requests.get(url, timeout=60)
response.raise_for_status()


with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
    with zip_file.open("SMSSpamCollection") as file:
        df = pd.read_csv(
            file,
            sep="\t",
            header=None,
            names=["label", "message"],
            encoding="utf-8"
        )

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst five records:")
display(df.head())

Dataset loaded successfully!
Dataset shape: (5572, 2)

Column names: ['label', 'message']

First five records:


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


# **Dataset Inspection and Data Quality Analysis**

In [ ]:


import re
import html
import pandas as pd


print("Number of messages:", len(df))
print("Number of columns:", len(df.columns))


print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())


print("\nMessage class distribution:")
print(df["label"].value_counts())

print("\nClass distribution in percentage:")
print((df["label"].value_counts(normalize=True) * 100).round(2))

df["character_count"] = df["message"].astype(str).str.len()
df["word_count"] = df["message"].astype(str).str.split().str.len()

print("\nAverage message length (characters):",
      round(df["character_count"].mean(), 2))

print("Average message length (words):",
      round(df["word_count"].mean(), 2))


print("\nDuplicate messages:", df["message"].duplicated().sum())

messages = df["message"].fillna("").astype(str)

print("\nMessages containing URLs:",
      messages.str.contains(r"https?://|www\.", case=False, regex=True).sum())

print("Messages containing HTML entities:",
      messages.str.contains(r"&(?:amp|lt|gt|quot|apos|nbsp);",
                            case=False, regex=True).sum())

print("Messages containing emojis or non-ASCII characters:",
      messages.str.contains(r"[^\x00-\x7F]", regex=True).sum())

print("Messages containing punctuation:",
      messages.str.contains(r"[^\w\s]", regex=True).sum())

print("Messages containing uppercase letters:",
      messages.str.contains(r"[A-Z]", regex=True).sum())


print("\nFirst five messages:")
display(df[["label", "message"]].head())

Number of messages: 5572
Number of columns: 2

Data types:
label      object
message    object
dtype: object

Missing values:
label      0
message    0
dtype: int64

Message class distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

Class distribution in percentage:
label
ham     86.59
spam    13.41
Name: proportion, dtype: float64

Average message length (characters): 80.49
Average message length (words): 15.6

Duplicate messages: 403

Messages containing URLs: 108
Messages containing HTML entities: 309
Messages containing emojis or non-ASCII characters: 483
Messages containing punctuation: 5109
Messages containing uppercase letters: 5463

First five messages:


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


# **Text Normalization and Cleaning**

In [ ]:


def clean_text(text):

    text = "" if pd.isna(text) else str(text)


    text = html.unescape(text)


    text = re.sub(r"<[^>]+>", " ", text)

    text = re.sub(r"https?://\S+|www\.\S+", " ", text,
                  flags=re.IGNORECASE)


    text = text.lower()


    text = re.sub(r"\s+", " ", text).strip()

    return text



df["clean_message"] = df["message"].apply(clean_text)

print("Text normalization completed successfully!")


display(df[["message", "clean_message"]].head(10))

Text normalization completed successfully!


,message,clean_message
0,"Go until jurong point, crazy.. Available only ...","go until jurong point, crazy.. available only ..."
1,Ok lar... Joking wif u oni...,ok lar... joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,U dun say so early hor... U c already then say...,u dun say so early hor... u c already then say...
4,"Nah I don't think he goes to usf, he lives aro...","nah i don't think he goes to usf, he lives aro..."
5,FreeMsg Hey there darling it's been 3 week's n...,freemsg hey there darling it's been 3 week's n...
6,Even my brother is not like to speak with me. ...,even my brother is not like to speak with me. ...
7,As per your request 'Melle Melle (Oru Minnamin...,as per your request 'melle melle (oru minnamin...
8,WINNER!! As a valued network customer you have...,winner!! as a valued network customer you have...
9,Had your mobile 11 months or more? U R entitle...,had your mobile 11 months or more? u r entitle...


# **Tokenization and Stop-Word Removal**

In [ ]:

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")


nltk_stop_words = set(stopwords.words("english"))


spacy_stop_words = nlp.Defaults.stop_words


sample_messages = df["clean_message"].head(5).tolist()

results = []

for i, message in enumerate(sample_messages, start=1):


    nltk_tokens = nltk.word_tokenize(message)

    nltk_filtered = [
        token for token in nltk_tokens
        if token.isalpha()
        and token.lower() not in nltk_stop_words
    ]


    doc = nlp(message)

    spacy_tokens = [token.text for token in doc]

    spacy_filtered = [
        token.text for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
        and token.is_alpha
    ]

    results.append({
        "Message No.": i,
        "NLTK Tokens Before": len(nltk_tokens),
        "NLTK Tokens After": len(nltk_filtered),
        "spaCy Tokens Before": len(spacy_tokens),
        "spaCy Tokens After": len(spacy_filtered),
        "NLTK Filtered Tokens": nltk_filtered,
        "spaCy Filtered Tokens": spacy_filtered
    })

token_comparison = pd.DataFrame(results)

display(token_comparison)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,Message No.,NLTK Tokens Before,NLTK Tokens After,spaCy Tokens Before,spaCy Tokens After,NLTK Filtered Tokens,spaCy Filtered Tokens
0,1,24,16,24,15,"[go, jurong, point, crazy, available, bugis, n...","[jurong, point, crazy, available, bugis, n, gr..."
1,2,8,6,8,6,"[ok, lar, joking, wif, u, oni]","[ok, lar, joking, wif, u, oni]"
2,3,36,20,31,15,"[free, entry, wkly, comp, win, fa, cup, final,...","[free, entry, wkly, comp, win, fa, cup, final,..."
3,4,13,9,13,6,"[u, dun, say, early, hor, u, c, already, say]","[u, dun, early, hor, u, c]"
4,5,15,7,15,5,"[nah, think, goes, usf, lives, around, though]","[nah, think, goes, usf, lives]"


# **Apply Stemming and Lemmatization**

In [ ]:

porter = PorterStemmer()
snowball = SnowballStemmer("english")
wordnet_lemmatizer = WordNetLemmatizer()


sample_words = [
    "playing", "played", "plays",
    "studies", "studying", "cars",
    "running", "better", "leaves", "easily"
]


comparison = []

for word in sample_words:

    porter_stem = porter.stem(word)
    snowball_stem = snowball.stem(word)


    nltk_lemma = wordnet_lemmatizer.lemmatize(word)


    spacy_doc = nlp(word)
    spacy_lemma = spacy_doc[0].lemma_

    comparison.append({
        "Original Word": word,
        "Porter Stem": porter_stem,
        "Snowball Stem": snowball_stem,
        "NLTK Lemma": nltk_lemma,
        "spaCy Lemma": spacy_lemma
    })

stemming_lemma_results = pd.DataFrame(comparison)

display(stemming_lemma_results)

,Original Word,Porter Stem,Snowball Stem,NLTK Lemma,spaCy Lemma
0,playing,play,play,playing,play
1,played,play,play,played,play
2,plays,play,play,play,play
3,studies,studi,studi,study,study
4,studying,studi,studi,studying,study
5,cars,car,car,car,car
6,running,run,run,running,run
7,better,better,better,better,well
8,leaves,leav,leav,leaf,leave
9,easily,easili,easili,easily,easily


# **Part-of-Speech Tagging and Named Entity Recognition**

In [ ]:

nltk.download("averaged_perceptron_tagger_eng")

sample_messages = df["clean_message"].head(5).tolist()

pos_results = []
entity_results = []

for i, message in enumerate(sample_messages, start=1):


    nltk_tokens = nltk.word_tokenize(message)
    nltk_pos = nltk.pos_tag(nltk_tokens)

    pos_results.append({
        "Message No.": i,
        "NLTK POS Tags": nltk_pos
    })


    doc = nlp(message)

    spacy_pos = [
        (token.text, token.pos_, token.tag_)
        for token in doc
        if not token.is_space
    ]

    pos_results[-1]["spaCy POS Tags"] = spacy_pos


    for ent in doc.ents:
        entity_results.append({
            "Message No.": i,
            "Entity": ent.text,
            "Entity Label": ent.label_,
            "Description": spacy.explain(ent.label_)
        })


print("POS Tagging Comparison:")
display(pd.DataFrame(pos_results))


print("\nNamed Entities Detected:")
if entity_results:
    display(pd.DataFrame(entity_results))
else:
    print("No named entities were detected in the selected messages.")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


POS Tagging Comparison:


,Message No.,NLTK POS Tags,spaCy POS Tags
0,1,"[(go, VB), (until, IN), (jurong, JJ), (point, ...","[(go, VERB, VB), (until, ADP, IN), (jurong, PR..."
1,2,"[(ok, JJ), (lar, NN), (..., :), (joking, VBG),...","[(ok, INTJ, UH), (lar, NOUN, NN), (..., PUNCT,..."
2,3,"[(free, JJ), (entry, NN), (in, IN), (2, CD), (...","[(free, ADJ, JJ), (entry, NOUN, NN), (in, ADP,..."
3,4,"[(u, JJ), (dun, NNS), (say, VBP), (so, RB), (e...","[(u, PRON, PRP), (dun, PROPN, NNP), (say, VERB..."
4,5,"[(nah, NN), (i, NNS), (do, VBP), (n't, RB), (t...","[(nah, INTJ, UH), (i, PRON, PRP), (do, AUX, VB..."



Named Entities Detected:


,Message No.,Entity,Entity Label,Description
0,3,2,CARDINAL,Numerals that do not fall under another type
1,3,21st may 2005,DATE,Absolute or relative dates or periods


# **Comparative Evaluation of NLTK and spaCy**

In [ ]:


comparison_results = []

sample_messages = df["clean_message"].head(5).tolist()

for i, message in enumerate(sample_messages, start=1):


    nltk_tokens = nltk.word_tokenize(message)

    nltk_filtered = [
        token for token in nltk_tokens
        if token.isalpha()
        and token.lower() not in nltk_stop_words
    ]

    nltk_lemmas = [
        wordnet_lemmatizer.lemmatize(token)
        for token in nltk_filtered
    ]


    doc = nlp(message)

    spacy_tokens = [token.text for token in doc]

    spacy_filtered = [
        token for token in doc
        if token.is_alpha and not token.is_stop
    ]

    spacy_lemmas = [token.lemma_ for token in spacy_filtered]

    comparison_results.append({
        "Message No.": i,
        "Original Message": message,
        "NLTK Token Count": len(nltk_tokens),
        "spaCy Token Count": len(spacy_tokens),
        "NLTK Filtered Tokens": nltk_filtered,
        "spaCy Filtered Tokens": [t.text for t in spacy_filtered],
        "NLTK Lemmas": nltk_lemmas,
        "spaCy Lemmas": spacy_lemmas
    })

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)

,Message No.,Original Message,NLTK Token Count,spaCy Token Count,NLTK Filtered Tokens,spaCy Filtered Tokens,NLTK Lemmas,spaCy Lemmas
0,1,"go until jurong point, crazy.. available only ...",24,24,"[go, jurong, point, crazy, available, bugis, n...","[jurong, point, crazy, available, bugis, n, gr...","[go, jurong, point, crazy, available, bugis, n...","[jurong, point, crazy, available, bugis, n, gr..."
1,2,ok lar... joking wif u oni...,8,8,"[ok, lar, joking, wif, u, oni]","[ok, lar, joking, wif, u, oni]","[ok, lar, joking, wif, u, oni]","[ok, lar, joke, wif, u, oni]"
2,3,free entry in 2 a wkly comp to win fa cup fina...,36,31,"[free, entry, wkly, comp, win, fa, cup, final,...","[free, entry, wkly, comp, win, fa, cup, final,...","[free, entry, wkly, comp, win, fa, cup, final,...","[free, entry, wkly, comp, win, fa, cup, final,..."
3,4,u dun say so early hor... u c already then say...,13,13,"[u, dun, say, early, hor, u, c, already, say]","[u, dun, early, hor, u, c]","[u, dun, say, early, hor, u, c, already, say]","[u, dun, early, hor, u, c]"
4,5,"nah i don't think he goes to usf, he lives aro...",15,15,"[nah, think, goes, usf, lives, around, though]","[nah, think, goes, usf, lives]","[nah, think, go, usf, life, around, though]","[nah, think, go, usf, live]"


### Observations and Discussion

The comparison demonstrates that NLTK and spaCy can produce different tokenization, stop-word-removal, and lemmatization results for the same SMS messages.

1. **Tokenization:** Differences may occur because the libraries handle punctuation, contractions, and special text patterns differently.
2. **Stop-word removal:** The resulting tokens can differ because the libraries use different stop-word lists and filtering rules.
3. **Lemmatization:** NLTK's WordNet lemmatizer and spaCy may produce different base forms because of their different linguistic approaches.
4. **Informal language:** SMS abbreviations, spelling variations, and incomplete sentences can affect both systems.
5. **Limitations:** This experiment compares outputs but does not establish which library is more accurate because no manually annotated reference output is used.

Overall, both libraries are useful for text preprocessing. The appropriate choice depends on the requirements of the application and the quality of the resulting text.

## Step 14: Final Analysis and Conclusion

### Description

The final analysis summarizes the results obtained from dataset inspection, text normalization, tokenization, stop-word removal, stemming, lemmatization, POS tagging, and named entity recognition.

It also discusses the limitations of the implemented preprocessing pipeline and identifies possible improvements for processing real-world SMS messages.

### Objective

To summarize the experimental results, evaluate the usefulness of the preprocessing techniques, and document the limitations of the system.

### Analysis

1. **Dataset inspection:** The SMS Spam Collection was loaded and examined to determine its size, class distribution, message lengths, and data quality.
2. **Text normalization:** A reusable function was implemented to decode HTML entities, remove HTML tags and URLs, convert text to lowercase, and normalize whitespace.
3. **Tokenization:** NLTK and spaCy were used to divide SMS messages into tokens.
4. **Stop-word removal:** Stop-word filtering reduced the number of tokens, although some common words may still be important for meaning and sentiment.
5. **Stemming and lemmatization:** Stemming produced simplified word forms, while lemmatization attempted to produce dictionary base forms.
6. **POS tagging:** Grammatical categories were assigned to tokens using NLTK and spaCy.
7. **Named entity recognition:** spaCy was used to identify and classify named entities in selected messages.
8. **Comparative evaluation:** Outputs from NLTK and spaCy were compared to identify differences in tokenization, filtering, and lemmatization.

### Limitations

- Informal abbreviations, spelling errors, and incomplete SMS sentences may reduce linguistic accuracy.
- Some URLs or HTML patterns may not be detected by the cleaning rules.
- Stop-word removal may remove meaningful words, including negations.
- Stemming may produce forms that are not valid dictionary words.
- NER may miss entities or assign incorrect labels to informal text.
- The comparison uses sample outputs and does not include a manually annotated benchmark for measuring accuracy.

### Future Improvements

- Expand the cleaning rules to handle SMS abbreviations and common spelling variations.
- Preserve negation words for sentiment-sensitive applications.
- Use POS-aware lemmatization and evaluate it on representative examples.
- Evaluate NER using manually verified annotations.
- Protect message privacy by avoiding unnecessary disclosure of personal information.

### Conclusion

The experiment demonstrated a text preprocessing pipeline for the SMS Spam Collection using NLTK and spaCy. The implemented techniques included normalization, tokenization, stop-word removal, stemming, lemmatization, POS tagging, and named entity recognition.

The comparison highlighted differences between the two NLP libraries and showed that preprocessing decisions influence the resulting text. The experiment also demonstrated that informal language and ambiguous expressions can affect the performance of NLP tools. Therefore, preprocessing methods should be selected and evaluated according to the requirements of the intended application.

In [ ]:


print("FINAL EXPERIMENT SUMMARY")
print("=" * 40)

print("Total SMS messages:", len(df))
print("Total columns:", len(df.columns))

print("\nMessage labels:")
print(df["label"].value_counts())

print("\nAverage message length:")
print(round(df["character_count"].mean(), 2), "characters")
print(round(df["word_count"].mean(), 2), "words")

print("\nMissing messages:", df["message"].isna().sum())
print("Duplicate messages:", df["message"].duplicated().sum())

print("\nSample original and cleaned messages:")
display(df[["message", "clean_message"]].head(5))

print("\nExperiment summary completed.")

FINAL EXPERIMENT SUMMARY
Total SMS messages: 5572
Total columns: 5

Message labels:
label
ham     4825
spam     747
Name: count, dtype: int64

Average message length:
80.49 characters
15.6 words

Missing messages: 0
Duplicate messages: 403

Sample original and cleaned messages:


,message,clean_message
0,"Go until jurong point, crazy.. Available only ...","go until jurong point, crazy.. available only ..."
1,Ok lar... Joking wif u oni...,ok lar... joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,U dun say so early hor... U c already then say...,u dun say so early hor... u c already then say...
4,"Nah I don't think he goes to usf, he lives aro...","nah i don't think he goes to usf, he lives aro..."



Experiment summary completed.


## Step 15: Viva Voce Questions and Answers

### 1. Why is lemmatization more linguistically meaningful than stemming?

Lemmatization uses linguistic information to convert a word into its dictionary base form. Stemming usually removes prefixes or suffixes using rules and may produce a form that is not a valid word.

### 2. When should stop words be retained?

Stop words should be retained when they contribute to the meaning of a sentence, such as in sentiment analysis, question answering, and tasks involving negation. For example, removing "not" can reverse the intended meaning.

### 3. How do POS tags support lemmatization?

Part-of-Speech tags identify the grammatical role of a word, such as noun, verb, or adjective. This information helps a lemmatizer choose an appropriate base form.

### 4. What causes NER errors in domain-specific text?

NER errors can occur because of unfamiliar names, domain-specific terminology, abbreviations, spelling variations, informal language, and incomplete sentences. A model trained on general text may not recognize specialized entities accurately.